# D01 — Rules conformance and handoff (A6-equivalent)

**Service:** `curalina_design_rules` &nbsp;|&nbsp; **Phase:** A6-equivalent (conformance notebook + handoff)

Every cell below either narrates, plots, or calls into `curalina_design_rules`
(via its stable `api` module or the `spatial` functions it wraps). No cell
contains rule logic — logic that does not yet exist in the package is a
finding recorded in Section 5/6, not something written here.

---

## Read this before the numbers below

`agent_instructions/00_design_rules_engine.md` and
`agentic_flow/12_design_rules_engine.md` describe this phase as reproducing
"the four golden scenarios in Design Manual §8" — four worked examples the
client authored specifically as an acceptance suite ("a gift").

**That text is not available.** `Training_Doc_1_-_Design_Manual.pdf` is
referenced throughout `agentic_flow/` but is not checked into this
repository at any path (verified by full-tree search, see
`ai_services/work_packets/RULES-A3-01.md`, "Known blockers"). Only
unrelated marketing/quiz PDFs exist under `attached_assets/`.

Consequently **this notebook makes no §8 conformance claim**. It runs the
package against the four *representative* fixtures in
`tests/fixtures/golden_scenarios.py` — scenarios built only from thresholds
already present in `rules/spatial_rules.yaml`, explicitly labelled
representative everywhere their results appear below — and it reports
honestly on what that run does and does not prove.

## 0. Manifest

In [1]:
import importlib.metadata
import json
import platform
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

# --- development shim: fixtures currently live under tests/, not package
# data (see RULES-A3-01.md). Delete this line at extraction, once/if the
# real §8 fixtures are promoted into a package-shipped golden set. ---
sys.path.insert(0, str(Path.cwd().parent / "tests"))
# ---------------------------------------------------------------------

RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = Path("runs") / f"D01_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
SEED = 20260912

import random

random.seed(SEED)

from curalina_design_rules import (
    evaluate_spatial_layout,
    load_rules,
    pinned_rules_version,
)

rules_version = pinned_rules_version()
rules = load_rules(rules_version)

def _pkg_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"

manifest = {
    "run_id": RUN_ID,
    "notebook": "D01_rules_conformance",
    "timestamp_utc": datetime.now(UTC).isoformat(),
    "git_sha": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True,
                               cwd=Path.cwd().parent).stdout.strip() or "uncommitted",
    "git_dirty": bool(subprocess.run(["git", "status", "--porcelain"], capture_output=True,
                                      text=True, cwd=Path.cwd().parent).stdout.strip()),
    "seed": SEED,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "rules_version": rules_version,
    "snapshot_id": None,  # no catalogue snapshot: this notebook exercises spatial
                          # geometry only, not catalogue-backed products (OQ-011)
    "packages": {
        "curalina_design_rules": _pkg_version("curalina-design-rules"),
        "shapely": _pkg_version("shapely"),
        "PyYAML": _pkg_version("PyYAML"),
    },
    "hardware": "CPU-only, no GPU/accelerator required",
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
manifest

{'run_id': '20260913T022937Z',
 'notebook': 'D01_rules_conformance',
 'timestamp_utc': '2026-09-13T02:29:37.596849+00:00',
 'git_sha': '6c35d20af5e2fad6b58c05661f4f05e569374d52',
 'git_dirty': True,
 'seed': 20260912,
 'python': '3.14.0',
 'platform': 'macOS-26.6.2-x86_64-i386-64bit-Mach-O',
 'rules_version': '2026.09.12-a',
 'snapshot_id': None,
 'packages': {'curalina_design_rules': '0.0.0',
  'shapely': '2.1.2',
  'PyYAML': '6.0.3'},
 'hardware': 'CPU-only, no GPU/accelerator required'}

## 1. Purpose and exit criteria

**What this notebook was asked to prove:** that `curalina_design_rules`
reproduces Design Manual §8's four worked scenario compositions exactly, as
the conformance gate for handoff to the recommendation and room-generation
consumers.

**What it can actually prove, given the inputs that exist:**

- That the A3 consumer-facing entry point (`evaluate_spatial_layout`) and
  the named A2b spatial checks it wraps run end-to-end, deterministically,
  against four *representative* scenarios spanning all three home
  categories (condo/mid/large) and three room types (living/dining/bedroom).
- That a scenario built to violate a named hard-spatial rule
  (`LR_FLOATING_ANCHOR`) is actually caught, and one built to pass is
  actually collision-free and within the walkway minimum.
- That `CMR_VALIDATION` surfaces as `needs_input` citing `OQ-001` on every
  scenario, exactly as the package promises, never as a silent pass.

**What it cannot prove:**

- §8 conformance. There is nothing to conform *to* — the source text was
  never supplied. Any result below that looks like a "pass" is a pass
  against YAML-derived thresholds already known to the engine, not against
  the client's worked examples.
- Anything about the style/material/colour rule families that `12_design_
  rules_engine.md` also scopes into §8's compositions (edge-radius ratios,
  material-mass ratios, palette nodal sampling). `style.py` and
  `palette.py` do not exist yet in `curalina_design_rules`
  (only `loader.py`, `spatial/`, `types/`, and `api.py` are built as of
  this run) — those rule families are blocked by `OQ-002`, `OQ-004`,
  `OQ-005`, `OQ-006`, `OQ-007`, `OQ-008` at the *design* level, not merely
  unexercised here.

**Exit criteria set before running:** a clean pass on every representative
scenario's *intended* result (collision-free where expected, the deliberate
violation actually firing, CMR always `needs_input`) is necessary but not
sufficient for any conformance claim above `accept-with-limitations`. A
`needs_input` firing where the manual requires a firm pass/fail
(`CMR_VALIDATION`) caps the decision at `accept-with-limitations` by
definition — it can never promote to a clean accept while OQ-001 is open,
regardless of how the spatial checks perform.

## 2. Inputs

Loaded through package adapters only — `curalina_design_rules.load_rules`
for the frozen rule tables, and the `tests/fixtures/golden_scenarios`
fixture module for the four representative scenarios (there is no ad-hoc
YAML/JSON parsing in this notebook).

In [2]:
from fixtures.golden_scenarios import ALL_SCENARIOS

print(f"rules_version pinned for this run: {rules_version}")
print(f"scenarios loaded: {len(ALL_SCENARIOS)}\n")
for s in ALL_SCENARIOS:
    print(f"  [{s.scenario_id}]")
    print(f"    room_type={s.room.room_type.value:12s} home_category={s.room.home_category.value:6s} "
          f"placements={len(s.placements)} expect_passes_hard_spatial={s.expect_passes_hard_spatial}")

rules_version pinned for this run: 2026.09.12-a
scenarios loaded: 4

  [condo_living_room_conversation_circle]
    room_type=living_room  home_category=condo  placements=2 expect_passes_hard_spatial=True
  [mid_dining_room_service_perimeter]
    room_type=dining_room  home_category=mid    placements=2 expect_passes_hard_spatial=True
  [large_living_room_floating_anchor_violation]
    room_type=living_room  home_category=large  placements=1 expect_passes_hard_spatial=False
  [condo_bedroom_pinched_ensuite_path]
    room_type=bedroom      home_category=condo  placements=1 expect_passes_hard_spatial=True


## 3. Execution — package calls only

Every scenario is run through the A3 entry point
`evaluate_spatial_layout` (collision + walkway + CMR), plus the two named
A2b checks the fixture module documents as needing extra points the entry
point does not have on its own: `check_lr_floating_anchor` for the large
living room, and `check_br_ensuite_path` for the pinched-corridor bedroom.
No arithmetic or rule logic is written in this cell — every check is a
function call into `curalina_design_rules`.

In [3]:
from curalina_design_rules.spatial import (
    check_br_ensuite_path,
    check_lr_floating_anchor,
)
from curalina_design_rules.types import Point

results = {}
for s in ALL_SCENARIOS:
    result = evaluate_spatial_layout(s.placements, s.room, rules)
    named_violations = ()

    if s.scenario_id in ("condo_living_room_conversation_circle",
                          "large_living_room_floating_anchor_violation"):
        room_width_mm = s.room.boundary[1].x_mm - s.room.boundary[0].x_mm
        sofa_placement = s.placements[0]
        named_violations += check_lr_floating_anchor(
            room_width_mm, sofa_placement.y_mm, rules.spatial_rule_table, "inst_sofa"
        )

    if s.scenario_id == "condo_bedroom_pinched_ensuite_path":
        bed_point = Point(10, 10)
        ensuite_point = Point(s.room.boundary[1].x_mm - 10, 10)
        named_violations += check_br_ensuite_path(
            s.placements, s.room, bed_point, ensuite_point,
            rules.spatial_rule_table, "inst_bed",
        )

    results[s.scenario_id] = {"scenario": s, "api_result": result, "named_violations": named_violations}

print("executed", len(results), "scenarios against rules_version", rules.rules_version)

executed 4 scenarios against rules_version 2026.09.12-a


## 4. Metrics

Computed from the package's own `RuleResult`/`Violation` objects — no
inline scoring formula. Every scenario is labelled **representative**, not
§8, in the table itself so nobody downstream can quote this table as
manual conformance.

In [4]:
import pandas as pd  # presentation only -- never business logic

rows = []
for scenario_id, r in results.items():
    api_result = r["api_result"]
    all_violations = tuple(api_result.violations) + tuple(r["named_violations"])
    by_severity = {}
    for v in all_violations:
        by_severity[v.severity.value] = by_severity.get(v.severity.value, 0) + 1
    rows.append({
        "scenario_id": scenario_id,
        "label": "REPRESENTATIVE (not Design Manual §8)",
        "home_category": r["scenario"].room.home_category.value,
        "room_type": r["scenario"].room.room_type.value,
        "expected_passes_hard_spatial": r["scenario"].expect_passes_hard_spatial,
        "api_passes_hard": api_result.passes_hard,
        "n_violations": len(all_violations),
        "hard": by_severity.get("hard", 0),
        "soft": by_severity.get("soft", 0),
        "needs_input": by_severity.get("needs_input", 0),
        "rule_ids_fired": ", ".join(sorted({v.rule_id for v in all_violations})),
    })

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(RUN_DIR / "metrics.csv", index=False)
(RUN_DIR / "metrics.json").write_text(metrics_df.to_json(orient="records", indent=2))
metrics_df

,scenario_id,label,home_category,room_type,expected_passes_hard_spatial,api_passes_hard,n_violations,hard,soft,needs_input,rule_ids_fired
0,condo_living_room_conversation_circle,REPRESENTATIVE (not Design Manual §8),condo,living_room,True,False,2,1,0,1,"CMR_VALIDATION, MIN_WALKWAY"
1,mid_dining_room_service_perimeter,REPRESENTATIVE (not Design Manual §8),mid,dining_room,True,False,2,1,0,1,"CMR_VALIDATION, MIN_WALKWAY"
2,large_living_room_floating_anchor_violation,REPRESENTATIVE (not Design Manual §8),large,living_room,False,False,3,2,0,1,"CMR_VALIDATION, LR_FLOATING_ANCHOR, MIN_WALKWAY"
3,condo_bedroom_pinched_ensuite_path,REPRESENTATIVE (not Design Manual §8),condo,bedroom,True,False,3,2,0,1,"BR_ENSUITE_PATH, CMR_VALIDATION, MIN_WALKWAY"


In [5]:
# rules_version is recorded once at manifest time; every scenario in this run
# used the identical pinned version -- confirms determinism across the batch.
assert {r["api_result"].rules_version for r in results.values()} == {rules_version}
print(f"all {len(results)} scenarios ran under a single pinned rules_version: {rules_version}")

needs_input_rate = metrics_df["needs_input"].gt(0).mean()
print(f"fraction of scenarios carrying at least one needs_input violation: {needs_input_rate:.0%}")
print("(this is expected to be 100% -- CMR_VALIDATION/OQ-001 fires on every "
      "scenario by design, see spatial.check_cmr_validation)")

all 4 scenarios ran under a single pinned rules_version: 2026.09.12-a
fraction of scenarios carrying at least one needs_input violation: 100%
(this is expected to be 100% -- CMR_VALIDATION/OQ-001 fires on every scenario by design, see spatial.check_cmr_validation)


## 5. Failure analysis

Every violation retained and shown — nothing filtered. Two things are
"failures" in different senses here and they are kept separate: a
`needs_input` result (the engine correctly refusing to guess) is not the
same failure mode as a `hard` violation (the engine catching a bad
layout), and neither is cherry-picked out of the table above.

In [6]:
all_violation_rows = []
for scenario_id, r in results.items():
    all_violations = tuple(r["api_result"].violations) + tuple(r["named_violations"])
    for v in all_violations:
        all_violation_rows.append({
            "scenario_id": scenario_id,
            "rule_id": v.rule_id,
            "severity": v.severity.value,
            "message": v.message,
            "source_section": v.source_section,
            "open_question_id": v.open_question_id,
        })

violations_df = pd.DataFrame(all_violation_rows)
violations_df.to_csv(RUN_DIR / "all_violations.csv", index=False)
print(f"{len(violations_df)} total violations retained across {len(results)} scenarios "
      f"(none filtered out)\n")
violations_df

10 total violations retained across 4 scenarios (none filtered out)



,scenario_id,rule_id,severity,message,source_section,open_question_id
0,condo_living_room_conversation_circle,MIN_WALKWAY,hard,No continuous walkway of at least 813mm found,9.1,NaN
1,condo_living_room_conversation_circle,CMR_VALIDATION,needs_input,Circulation-to-Mass Ratio formula is undefined...,"9 implementation note, 10 STEP 5",OQ-001
2,mid_dining_room_service_perimeter,MIN_WALKWAY,hard,No continuous walkway of at least 914mm found,9.1,NaN
3,mid_dining_room_service_perimeter,CMR_VALIDATION,needs_input,Circulation-to-Mass Ratio formula is undefined...,"9 implementation note, 10 STEP 5",OQ-001
4,large_living_room_floating_anchor_violation,MIN_WALKWAY,hard,No continuous walkway of at least 1219mm found,9.1,NaN
5,large_living_room_floating_anchor_violation,CMR_VALIDATION,needs_input,Circulation-to-Mass Ratio formula is undefined...,"9 implementation note, 10 STEP 5",OQ-001
6,large_living_room_floating_anchor_violation,LR_FLOATING_ANCHOR,hard,Sofa is 0mm off the wall; rooms over the width...,9.2.2.3,NaN
7,condo_bedroom_pinched_ensuite_path,MIN_WALKWAY,hard,No continuous walkway of at least 813mm found,9.1,NaN
8,condo_bedroom_pinched_ensuite_path,CMR_VALIDATION,needs_input,Circulation-to-Mass Ratio formula is undefined...,"9 implementation note, 10 STEP 5",OQ-001
9,condo_bedroom_pinched_ensuite_path,BR_ENSUITE_PATH,hard,No continuous 1067mm path from bed to ensuite ...,9.4.4,NaN


In [7]:
# Every scenario's expectation is checked explicitly against what the package
# actually returned -- a mismatch here is a real finding, not something to
# average away.
mismatches = []
for scenario_id, r in results.items():
    scenario = r["scenario"]
    api_result = r["api_result"]
    # api_result.passes_hard is always False while CMR/OQ-001 is open (needs_input
    # counts as a hard-blocking severity) -- so we check the *non-CMR* hard result
    # against the fixture's stated expectation, which is what the fixture module
    # itself documents as the intent of expect_passes_hard_spatial.
    non_cmr_hard = [v for v in api_result.violations
                    if v.rule_id != "CMR_VALIDATION" and v.severity.value == "hard"]
    non_cmr_hard += [v for v in r["named_violations"] if v.severity.value == "hard"]
    actual_non_cmr_pass = len(non_cmr_hard) == 0
    if actual_non_cmr_pass != scenario.expect_passes_hard_spatial:
        mismatches.append((scenario_id, tuple(sorted({v.rule_id for v in non_cmr_hard}))))

print(f"{len(mismatches)} of {len(results)} scenarios mismatch their fixture module's "
      f"stated expect_passes_hard_spatial (CMR_VALIDATION set aside as its own "
      f"separate, always-needs_input finding -- see below):\n")
for scenario_id, rule_ids in mismatches:
    print(f"  MISMATCH  {scenario_id}: fixture docstring claims a pass, "
          f"engine additionally fired {rule_ids}")

print()
cmr_findings = violations_df[violations_df.rule_id == "CMR_VALIDATION"]
print(f"CMR_VALIDATION fired needs_input on {len(cmr_findings)} of {len(results)} "
      f"scenarios, every one citing {cmr_findings.open_question_id.unique().tolist()}.")
print("This is expected and by design -- see Section 1 exit criteria.")

3 of 4 scenarios mismatch their fixture module's stated expect_passes_hard_spatial (CMR_VALIDATION set aside as its own separate, always-needs_input finding -- see below):

  MISMATCH  condo_living_room_conversation_circle: fixture docstring claims a pass, engine additionally fired ('MIN_WALKWAY',)
  MISMATCH  mid_dining_room_service_perimeter: fixture docstring claims a pass, engine additionally fired ('MIN_WALKWAY',)
  MISMATCH  condo_bedroom_pinched_ensuite_path: fixture docstring claims a pass, engine additionally fired ('BR_ENSUITE_PATH', 'MIN_WALKWAY')

CMR_VALIDATION fired needs_input on 4 of 4 scenarios, every one citing ['OQ-001'].
This is expected and by design -- see Section 1 exit criteria.


### Unanticipated finding: `MIN_WALKWAY` fires on 3 of 4 "passing" fixtures

This is the kind of result Section 5 exists to surface rather than hide.

Three of the four representative fixtures carry docstrings asserting they
are "well within" the walkway minimum, or otherwise collision-free and
easily passing (`tests/fixtures/golden_scenarios.py`). Run through the
actual package code, all three additionally trip `MIN_WALKWAY` (and, for
the bedroom scenario, `BR_ENSUITE_PATH`) as **hard** violations — something
their authors evidently did not anticipate when writing "well within the
32" walkway minimum."

The reason is visible in `spatial/geometry.py`: `has_walkway` implements a
strict morphological-opening test — it requires that essentially **all**
free floor area survive an erosion by half the minimum width, not merely
that *a* path of sufficient width connects two points. Any residual sliver
of free space thinner than that (e.g. the ~254mm gap the condo living-room
fixture leaves between its sofa and the near wall) counts as "pinched" and
fails the whole-room check, even though the room plainly has ample
circulation elsewhere. `12_design_rules_engine.md` specifies exactly this
erode-then-dilate primitive for every §9 walkway/reachability rule, so this
looks like the engine doing what it was told, applied to fixtures that were
authored for a different purpose (exercising the OM sofa-to-table spacing
and credenza-proportion rules in the A3 packet) and not dimensioned with
this stricter whole-room reading in mind.

This is recorded as a genuine open finding, not smoothed into the metrics
table above: either (a) the fixtures need re-authoring with full-perimeter
clearance before they can demonstrate a walkway pass, or (b) `12_design_
rules_engine.md`'s "no pinched area anywhere" reading of §9.1 is stricter
than what a designer would consider a real failure, and is worth confirming
with the design authority. Neither (a) nor (b) is an existing `OQ-xxx`; it
is a new gap this run surfaced, and it is named as such in the decision
record below rather than folded into an existing open question it does not
match.

### Style / material / palette rule families — cannot be exercised at all

`12_design_rules_engine.md`'s §8 framing expects a conformance run to also
cover proportion, colour and material rules (the 70/30 edge rule, material
mass ratios, nodal palette sampling). As of this run, `curalina_design_
rules` has no `style.py` or `palette.py` module — only `loader.py`,
`types/`, `spatial/`, and `api.py` exist. This is not a gap this notebook
introduces; it reflects the package's actual build order
(`12_design_rules_engine.md` build order steps 6–7 are not yet built). It
is recorded here because a conformance notebook that stayed silent about an
entire missing rule family would be more misleading than one that names
it.

In [8]:
import importlib

for mod in ("curalina_design_rules.style", "curalina_design_rules.palette", "curalina_design_rules.pruning"):
    try:
        importlib.import_module(mod)
        status = "present"
    except ModuleNotFoundError:
        status = "NOT IMPLEMENTED"
    print(f"{mod:38s} {status}")

curalina_design_rules.style            NOT IMPLEMENTED
curalina_design_rules.palette          NOT IMPLEMENTED
curalina_design_rules.pruning          NOT IMPLEMENTED


## 6. Decision record

This cell is the gate artifact. It is written from the evidence above, not
the other way around.

In [9]:
open_questions_touched = {
    "OQ-001": "CMR_VALIDATION formula undefined -- fires needs_input on every "
              "scenario in this run (100%); blocks spatial validation step 5 and "
              "final validation step 11 per the Design Manual. Directly exercised "
              "by this notebook's Section 5.",
    "OQ-002": "OM 70/30 edge-radius rule denominator undefined -- blocks OM style "
              "validation. Not exercised: style.py does not exist yet.",
    "OQ-004": "Material composition ratio measurement basis undefined (area vs "
              "count vs footprint vs visual mass) -- blocks material composition "
              "validation. Not exercised: style.py does not exist yet.",
    "OQ-007": "Anchor colour hex library missing -- blocks palette generation and "
              "wall-paint recalibration. Not exercised: palette.py does not exist yet.",
}

new_finding_not_yet_an_oq = (
    "MIN_WALKWAY fires as a hard violation on 3 of 4 representative fixtures "
    "whose docstrings assert an easy pass (see the cell above). Root cause: "
    "has_walkway's strict 'no pinched free area anywhere' semantics vs. fixtures "
    "authored for a different rule family without full-perimeter clearance. Not "
    "an existing OQ-xxx -- flagged here as a new gap for the design authority "
    "and/or fixture author to resolve, not folded into the metrics as a pass."
)

decision = {
    "run_id": RUN_ID,
    "gate": "D01 / A6-equivalent (rules-engine conformance and handoff)",
    "decision": "accept-with-limitations",
    "rationale": (
        "The A3 consumer-facing entry point and the A2b spatial checks it wraps "
        "run deterministically end to end against four representative scenarios, "
        "and CMR_VALIDATION surfaced needs_input with the correct OQ-001 citation "
        "on every scenario, never a silent pass -- that part of the engine behaves "
        "exactly as designed. "
        "\n\n"
        "However, this run's own results do NOT cleanly match the fixture module's "
        "documented expectations: only 1 of 4 scenarios (the deliberate "
        "large_living_room_floating_anchor_violation failure case) matches its "
        "stated expect_passes_hard_spatial once CMR is set aside. The other 3 "
        "additionally trip MIN_WALKWAY (and, for the bedroom case, "
        "BR_ENSUITE_PATH) as hard violations despite being documented as passing -- "
        "see the finding recorded above. This is evidence the engine's spatial "
        "primitives run correctly against *some* interpretation of the rules, but "
        "it is NOT evidence that the representative fixtures demonstrate a clean "
        "pass, and it must not be reported as one. "
        "\n\n"
        "Separately and independently: this is NOT evidence of Design Manual "
        "section 8 conformance, because section 8's text (the client's four "
        "worked examples) is not present anywhere in this repository -- verified "
        "by full-tree search, see RULES-A3-01.md 'Known blockers'. The scenarios "
        "exercised here are representative fixtures built from already-loaded "
        "YAML thresholds, explicitly labelled as such everywhere they appear in "
        "this notebook. A conformance claim against section 8 cannot be made -- "
        "accepted, rejected, or otherwise -- until the Design Manual (or an "
        "authoritative extract of its section 8) is supplied. "
        "\n\n"
        "Further separately: this run cannot say anything about the "
        "style/material/palette rule families the manual's section 8 "
        "compositions also depend on, because those modules (style.py, "
        "palette.py) do not exist yet in curalina_design_rules."
    ),
    "sample_size": {
        "scenarios": len(ALL_SCENARIOS),
        "home_categories_covered": sorted({s.room.home_category.value for s in ALL_SCENARIOS}),
        "room_types_covered": sorted({s.room.room_type.value for s in ALL_SCENARIOS}),
    },
    "blocked_by_design_manual_absence": True,
    "unanticipated_findings": [new_finding_not_yet_an_oq],
    "open_questions_touched": open_questions_touched,
    "open_questions_still_open_project_wide_not_exercised_here": [
        "OQ-003", "OQ-005", "OQ-006", "OQ-008", "OQ-009", "OQ-010", "OQ-011", "OQ-012",
    ],
    "what_would_promote_this_to_a_clean_accept": [
        "The Design Manual section 8 text (or an equivalent client-authored "
        "extract) is supplied and the real four worked examples are encoded "
        "and reproduced exactly.",
        "OQ-001 (CMR formula) is resolved so CMR_VALIDATION can return a firm "
        "pass/fail instead of needs_input.",
        "The MIN_WALKWAY mismatch above is resolved -- either by re-authoring "
        "the fixtures with full-perimeter clearance, or by design-authority "
        "confirmation that the strict whole-room opening test is the intended "
        "reading of section 9.1.",
        "style.py and palette.py are built so material/colour rules can be "
        "exercised in the same conformance run (their own OQ-002/004/005/006/"
        "007/008 blockers would still need resolving independently).",
    ],
    "what_would_force_a_no_go_instead": [
        "The deliberate failure scenario (large_living_room_floating_anchor_"
        "violation) failing to fire LR_FLOATING_ANCHOR (it did fire, in this run).",
        "CMR_VALIDATION returning a silent pass instead of needs_input (it did "
        "not, in this run).",
        "A collision (NO_PLACEMENT_ON_VIOLATION) appearing in any scenario not "
        "designed to have one (none appeared, in this run).",
    ],
}

(RUN_DIR / "decision_record.json").write_text(json.dumps(decision, indent=2))
print(json.dumps(decision, indent=2))

{
  "run_id": "20260913T022937Z",
  "gate": "D01 / A6-equivalent (rules-engine conformance and handoff)",
  "decision": "accept-with-limitations",
  "rationale": "The A3 consumer-facing entry point and the A2b spatial checks it wraps run deterministically end to end against four representative scenarios, and CMR_VALIDATION surfaced needs_input with the correct OQ-001 citation on every scenario, never a silent pass -- that part of the engine behaves exactly as designed. \n\nHowever, this run's own results do NOT cleanly match the fixture module's documented expectations: only 1 of 4 scenarios (the deliberate large_living_room_floating_anchor_violation failure case) matches its stated expect_passes_hard_spatial once CMR is set aside. The other 3 additionally trip MIN_WALKWAY (and, for the bedroom case, BR_ENSUITE_PATH) as hard violations despite being documented as passing -- see the finding recorded above. This is evidence the engine's spatial primitives run correctly against *some* int

---

### Handoff note for `ai-ml-lead`

This notebook is evidence, not a sign-off. It supports: the spatial
engine's consumer-facing wrapper (`evaluate_spatial_layout`) runs
deterministically end to end, and its `needs_input` handling for OQ-001 is
real, not decorative — it fired correctly on all four scenarios.

It does **not** support a clean pass. Two independent gaps keep this at
`accept-with-limitations`:

1. Only 1 of the 4 representative scenarios actually matched its own
   fixture-module docstring's stated expectation once CMR is set aside —
   the other 3 additionally trip `MIN_WALKWAY`/`BR_ENSUITE_PATH`, a genuine
   unanticipated finding recorded in Section 5/6, not smoothed away.
2. Any claim that `curalina_design_rules` reproduces Design Manual §8 is
   unsupportable, because §8 was never supplied to this repository.

Both gaps need a decision from the design authority — supply the §8 text
(or accept representative fixtures as a permanent substitute for that
gate), and confirm whether `has_walkway`'s whole-room "no pinch anywhere"
reading of §9.1 is intended or whether the fixtures should be re-authored
with full-perimeter clearance — before this can promote past
`accept-with-limitations`.

Run artifacts (`manifest.json`, `metrics.csv`/`.json`, `all_violations.csv`,
`decision_record.json`) are written to `RUN_DIR` for archival, per
`16_notebook_standard.md`.